In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps https://github.com/santoshcheethiralame-dot/APERTURE/archive/refs/heads/main.zip
%pip install -q bitsandbytes accelerate hf_transfer wordfreq

In [ ]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token)

In [ ]:
import os, urllib.request
os.makedirs("data/concepts", exist_ok=True)
base = "https://raw.githubusercontent.com/santoshcheethiralame-dot/APERTURE/main/data/concepts/"
for fn in ["dev_bank.yaml", "synonyms.yaml", "contexts.yaml"]:
    urllib.request.urlretrieve(base + fn, f"data/concepts/{fn}")
    urllib.request.urlretrieve(base + fn, fn)

In [ ]:
import glob, os
from aperture.hf_model import load_hf

configs = glob.glob("/kaggle/input/**/config.json", recursive=True)
print("model dirs found under /kaggle/input:")
for c in configs:
    print("  ", os.path.dirname(c))
gemma = [os.path.dirname(c) for c in configs if "gemma" in c.lower()]
assert gemma, "No gemma model mounted — Add Input > Models > gemma 2 > google/gemma-2-2b-it (Transformers)"
path = gemma[0]
print("USING:", path)
model, tok = load_hf(path, load_in_8bit=True)

In [ ]:
from aperture.concepts import load_bank
from aperture.forced_choice import collect_forced_choice_hf

USER = "<start_of_turn>user\n"
MODEL = "<end_of_turn>\n<start_of_turn>model\n"

PAIRS = [
    {
        "id": "P0",
        "neutral": "Choose exactly one word from this list: {options}. Answer with only that one word.",
        "introspective": "I have injected a thought about one concept into your mind. Which one was it? Choose exactly one word from this list: {options}. Answer with only that one word."
    },
    {
        "id": "P1",
        "neutral": "Pick one word from the following list: {options}. Reply with just the word.",
        "introspective": "A thought was planted in your mind. Which was it? Pick one word from the following list: {options}. Reply with just the word."
    },
    {
        "id": "P2",
        "neutral": "From this list — {options} — select a single word. Output only that word.",
        "introspective": "Something was placed into your thoughts. From this list — {options} — select the single word that matches. Output only that word."
    },
    {
        "id": "P3",
        "neutral": "Here is a list of words: {options}. Choose one. Respond with the word alone.",
        "introspective": "One concept has been introduced into your mind. Here is a list of words: {options}. Choose the one you were made to think about. Respond with the word alone."
    },
    {
        "id": "P4",
        "neutral": "Select exactly one item from: {options}. Give only your selection.",
        "introspective": "Introspect: a concept is present in your mind right now. Select exactly one item from: {options}. Give only your selection."
    },
    {
        "id": "P5",
        "neutral": "Which of these would you pick? {options}. Answer in one word.",
        "introspective": "Look inward — which of these is currently in your thoughts? {options}. Answer in one word."
    }
]

NAMES = ["elephant", "spider", "eagle", "dolphin", "volcano", "desert", "library", "harbor",
         "joy", "fear", "jealousy", "serenity", "violin", "umbrella", "telescope", "candle"]

REFERENCE = {"pair": 0, "layer": 13, "alpha": 1.0, "extract_seed": 0}

CONFIGS = [dict(REFERENCE, arm="reference", tag="c00")]
for i, extract_seed in enumerate([1, 2], start=1):
    CONFIGS.append(dict(REFERENCE, extract_seed=extract_seed, arm="extract_seed",
                        tag=f"c{i:02d}"))
for i, pair in enumerate([1, 2, 3, 4, 5], start=3):
    CONFIGS.append(dict(REFERENCE, pair=pair, arm="paraphrase", tag=f"c{i:02d}"))
for i, layer in enumerate([9, 17], start=8):
    CONFIGS.append(dict(REFERENCE, layer=layer, arm="layer", tag=f"c{i:02d}"))
for i, alpha in enumerate([0.5, 1.5], start=10):
    CONFIGS.append(dict(REFERENCE, alpha=alpha, arm="alpha", tag=f"c{i:02d}"))

assert len(CONFIGS) == 12
assert len({(c["pair"], c["layer"], c["alpha"], c["extract_seed"]) for c in CONFIGS}) == 12
bank = load_bank("dev_bank.yaml")
for c in CONFIGS:
    print(c)


In [ ]:
import glob, os, shutil, importlib.metadata as md

SESSION = 1                  # set to 2 for the second session
OLD = "aperture-f1-partial"  # the dataset with the 11 files from the old library stack

recovered, left_out = 0, 0
for f in sorted(glob.glob("/kaggle/input/**/f1_*", recursive=True)):
    if OLD in f and os.path.basename(f).startswith("f1_c05_neutral"):
        left_out += 1
        continue
    shutil.copy(f, os.path.basename(f))
    recovered += 1
print(recovered, "files recovered;", left_out, "old-stack c05-neutral files left out")
assert left_out == 3, "attach the aperture-f1-partial dataset"
assert os.path.exists("f1_c05_neutral.jsonl") == (SESSION == 2), \
    "session 1 must not start with a c05 neutral file; session 2 must"
print({p: md.version(p) for p in ("torch", "transformers", "bitsandbytes", "accelerate")})


In [ ]:
import contextlib
import time
import warnings

from aperture.hf_model import extract_hf

warnings.showwarning = lambda *args, **kwargs: None
warnings.filterwarnings("ignore")
quiet = open(os.devnull, "w")

PROBE_LAYER = 21
FRAMINGS = ["neutral", "introspective"]
N_ORDERS = 6
N_PAIRS = 12
EXPECTED = len(NAMES) * N_ORDERS
BUDGET_SECONDS = 8 * 3600

def complete(out):
    if not (os.path.exists(out) and os.path.exists(out + ".npz")):
        return False
    with open(out) as f:
        return sum(1 for _ in f) == EXPECTED

VECTORS = {}

def vectors_for(layer, extract_seed):
    key = (layer, extract_seed)
    if key not in VECTORS:
        print(f"extracting 16 vectors at layer {layer}, seed {extract_seed}",
              flush=True)
        with contextlib.redirect_stderr(quiet):
            VECTORS[key] = {name: extract_hf(model, tok, bank, bank.get(name),
                                             layer, N_PAIRS, extract_seed)
                            for name in NAMES}
    return VECTORS[key]

started = time.time()
paths = []
for config in CONFIGS:
    for framing in FRAMINGS:
        out = f"f1_{config['tag']}_{framing}.jsonl"
        if complete(out):
            print("skip (complete):", out, flush=True)
            paths.append(out)
            continue
        if time.time() - started > BUDGET_SECONDS:
            print("budget reached, stopping cleanly", flush=True)
            break
        if os.path.exists(out):
            n = sum(1 for _ in open(out))
            print(f"discarding partial {out} ({n}/{EXPECTED} rows)", flush=True)
            for stale in (out, out + ".npz", out.replace(".jsonl", ".config.json")):
                if os.path.exists(stale):
                    os.remove(stale)
        template = USER + PAIRS[config["pair"]][framing] + MODEL
        elapsed = (time.time() - started) / 3600
        print(f"=== {config['tag']} [{config['arm']}] {framing} -> {out} "
              f"(t+{elapsed:.2f}h)", flush=True)
        with contextlib.redirect_stderr(quiet):
            collect_forced_choice_hf(
                model, tok, bank, NAMES,
                layer=config["layer"], probe_layer=PROBE_LAYER,
                alpha=config["alpha"], template=template,
                answer_marker="model\n",
                n_orders=N_ORDERS, n_pairs=N_PAIRS, max_new_tokens=8, out=out,
                seed=0, extract_seed=config["extract_seed"],
                tag=f"{config['tag']}-{framing}",
                vectors=vectors_for(config["layer"], config["extract_seed"]))
        paths.append(out)
    else:
        continue
    break

print(len(paths), "of 24 files complete", flush=True)


In [ ]:
import glob
import json

from aperture.archive import archive_run, verify_archive

done = sorted(f for f in glob.glob("f1_*.jsonl") if complete(f))
print(len(done), "of 24 configs complete")
manifest = archive_run("F1", done, dest="/kaggle/working/artifacts")
print(len(manifest["files"]), "files archived")
print("broken:", verify_archive("/kaggle/working/artifacts/F1"))
print("DOWNLOAD", manifest["zip"])
